# Week 3: Named Entity Extraction
## Key Deliverables
• EntityExtractor class with regex patterns for numeric entities 

• Amenity detection using Week 1 taxonomy

• Labeled dataset: 200-300 remarks with entity spans

• Evaluation script calculating precision/recall/F1

• Target: 85%+ F1 score on test set

• Error analysis identifying top failure patterns

In [ ]:
import sys
import os
import json
import re

sys.path.append(os.path.abspath('..')) 

from scripts.evaluate_entity import evaluate
from scripts.meaningful_taxonomy_json_builder import taxonomy_data

## 1. Baseline Entity Extractor
First, we build a baseline `EntityExtractor` using standard regular expressions. This initial version targets standard numeric digits and common abbreviations (e.g., "3 beds", "2 baths").

In [36]:
class DumbBaselineExtractor:
    def extract_spans(self, text):
        entities = []
        bed_patterns = [r'(\d+)\s*(?:bed|br|bedroom)s?', r'(\d+)bd']
        for pattern in bed_patterns:
            for match in re.finditer(pattern, text, re.I):
                entities.append({"start": match.start(), "end": match.end(), "label": "BEDROOMS"})
        return entities


## 2. Error Analysis
If the model did not reach the 85% target, we need to analyze the False Negatives (entities missed by the regex). The following script compares our predictions with the ground truth to identify failure patterns.

In [ ]:
template_file = "../data/processed/data_template.jsonl"
baseline_pred_file = "../data/processed/data_prediction_baseline.jsonl"

baseline_extractor = DumbBaselineExtractor()

with open(template_file, 'r', encoding='utf-8') as fin, \
     open(baseline_pred_file, 'w', encoding='utf-8') as fout:
    for line in fin:
        data = json.loads(line)
        spans = baseline_extractor.extract_spans(data["text"])
        fout.write(json.dumps({"id": data["id"], "text": data["text"], "entities": spans}) + '\n')

print("--- 1. Baseline output ---")
p, r, f1 = evaluate(template_file, baseline_pred_file)
print(f"Precision: {p:.3f} | Recall: {r:.3f} | F1 Score: {f1:.3f}")

--- 1. Baseline output ---
Precision: 1.000 | Recall: 0.040 | F1 Score: 0.077


: 

## 3. Findings & Model Improvement
Based on the Error Analysis, the initial regex missed entities primarily due to:
1. **Hyphenated formats**: e.g., `"4-bedroom"` (The `\s*` pattern failed to capture the hyphen).
2. **Spelled-out numbers**: e.g., `"six bedrooms"` (The `\d+` pattern only captures Arabic numerals).

**Solution**: 
I updated the `entity_extractor.py` script directly to handle these edge cases by modifying the regex to `r'(\d+|one|two|three|four|five|six|...)\s*-?\s*(?:bed|br|bedroom)s?'`. 
After modifying the source script, the F1 score improved

In [ ]:
from scripts.entity_extractor import EntityExtractor

improved_pred_file = "../data/processed/data_prediction_improved.jsonl"
extractor = EntityExtractor()

with open(template_file, 'r', encoding='utf-8') as fin, \
     open(improved_pred_file, 'w', encoding='utf-8') as fout:
    for line in fin:
        data = json.loads(line)
        predicted_spans = extractor.extract_spans(data["text"])
        fout.write(json.dumps({"id": data["id"], "text": data["text"], "entities": predicted_spans}) + '\n')

print("--- 2. Improved Version ---")
p, r, f1 = evaluate(template_file, improved_pred_file)
print(f"Precision: {p:.3f} | Recall: {r:.3f} | F1 Score: {f1:.3f}")

--- 2. Improved Version ---
Precision: 1.000 | Recall: 0.906 | F1 Score: 0.951


: 